In [ ]:
# -*- coding: utf-8 -*-
"""
Arabic Opinion Mining Pipeline
BERTopic + MARBERT Sentiment Analysis
Clean Version
"""

# ─────────────────────────────────────────────
# 0. INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
!pip install bertopic transformers sentence-transformers umap-learn hdbscan arabert pyarabic arabic-reshaper python-bidi -q

# ─────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────
import os
import re

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import pyarabic.araby as araby
import arabic_reshaper
from bidi.algorithm import get_display

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    roc_auc_score,
)

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ─────────────────────────────────────────────
# 2. GOOGLE DRIVE SETUP
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/arabic_opinion_clustering/'
os.makedirs(save_path, exist_ok=True)
print(f"Save directory ready: {save_path}")

# ─────────────────────────────────────────────
# 3. LOAD DATASET & STRATIFIED SAMPLE
# ─────────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/Arabic_tweets_sentiment.csv', sep='\t')
print(f"Full dataset shape: {df.shape}")
print(df['class'].value_counts())

df_sample, _ = train_test_split(
    df[['text', 'class']],
    train_size=100000,
    stratify=df['class'],
    random_state=42
)
print(f"\nSample shape: {df_sample.shape}")
print(df_sample['class'].value_counts())

df_sample.to_csv(save_path + 'df_sample.csv', index=False)
print("Saved df_sample.csv")

# ─────────────────────────────────────────────
# 4. PREPROCESSING
# ─────────────────────────────────────────────
def preprocess_arabic(text):
    text = re.sub(r'http\S+|www\S+', '', text)          # Remove URLs
    text = re.sub(r'@\w+|#\w+', '', text)               # Remove mentions and hashtags
    text = re.sub(r'\d+', '', text)                      # Remove numbers
    text = re.sub(r'[^\w\s\u0600-\u06FF]', '', text)    # Remove non-Arabic punctuation
    text = araby.strip_diacritics(text)                  # Remove diacritics
    text = araby.normalize_alef(text)                    # Normalize alef
    text = araby.normalize_teh(text)                     # Normalize teh
    text = re.sub(r'\s+', ' ', text).strip()            # Remove extra whitespace
    return text

df_sample = df_sample.dropna(subset=['text']).reset_index(drop=True)
df_sample['text'] = df_sample['text'].astype(str)
df_sample['clean_text'] = df_sample['text'].apply(preprocess_arabic)
df_sample = df_sample[df_sample['clean_text'].str.len() > 0].reset_index(drop=True)

print(f"After preprocessing: {df_sample.shape}")
print(df_sample[['text', 'clean_text']].head())

df_sample.to_csv(save_path + 'df_preprocessed.csv', index=False)
print("Saved df_preprocessed.csv")

# ─────────────────────────────────────────────
# 5. AraBERT EMBEDDINGS (CLS TOKEN)
# ─────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
!nvidia-smi

print("Loading AraBERT for embeddings...")
emb_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
emb_model = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv2")
emb_model.eval().to(device)

def get_embeddings(texts, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = emb_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            output = emb_model(**encoded)
            embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        if (i // batch_size) % 15 == 0:
            print(f"  Processed {i}/{len(texts)} tweets")
    return np.vstack(all_embeddings)

print("Generating embeddings...")
texts = df_sample['clean_text'].tolist()
embeddings = get_embeddings(texts)
print(f"Embeddings shape: {embeddings.shape}")

np.save(save_path + 'embeddings.npy', embeddings)
print("Saved embeddings.npy")

# ─────────────────────────────────────────────
# 6. UMAP DIMENSIONALITY REDUCTION
# ─────────────────────────────────────────────
print("Applying UMAP...")
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
reduced_embeddings = umap_model.fit_transform(embeddings)
print(f"Reduced embeddings shape: {reduced_embeddings.shape}")

np.save(save_path + 'reduced_embeddings.npy', reduced_embeddings)
print("Saved reduced_embeddings.npy")

# ─────────────────────────────────────────────
# 7. BERTopic TOPIC MODELING
# ─────────────────────────────────────────────
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language='multilingual',
    calculate_probabilities=True,
    verbose=True
)

print("Fitting BERTopic...")
topics, probs = topic_model.fit_transform(
    df_sample['clean_text'].tolist(),
    embeddings
)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_noise  = list(topics).count(-1)
print(f"Topics found:     {n_topics}")
print(f"Noise percentage: {n_noise / len(topics) * 100:.2f}%")

df_sample['topic'] = topics
df_sample.to_csv(save_path + 'df_topics.csv', index=False)
topic_model.save(save_path + 'bertopic_model')
print("Saved df_topics.csv and bertopic_model")

# ─────────────────────────────────────────────
# 8. OUTLIER REDUCTION
# ─────────────────────────────────────────────
print("Reducing outliers...")
new_topics = topic_model.reduce_outliers(
    df_sample['clean_text'].tolist(),
    topics,
    strategy="probabilities",
    probabilities=probs
)

n_noise_new  = list(new_topics).count(-1)
n_topics_new = len(set(new_topics)) - (1 if -1 in new_topics else 0)
print(f"Topics after outlier reduction: {n_topics_new}")
print(f"Noise after outlier reduction:  {n_noise_new / len(new_topics) * 100:.2f}%")

df_sample['topic'] = new_topics
topic_model.update_topics(
    df_sample['clean_text'].tolist(),
    topics=df_sample['topic'].tolist()
)

df_sample.to_csv(save_path + 'df_topics_reduced.csv', index=False)
print("Saved df_topics_reduced.csv")

# ─────────────────────────────────────────────
# 9. TOPIC INFO & INITIAL VISUALIZATION
# ─────────────────────────────────────────────
topic_info = topic_model.get_topic_info()
print(f"Total topics: {len(topic_info)}")
print(topic_info.head(20))

topic_counts = df_sample[df_sample['topic'] != -1]['topic'].value_counts()
top10_ids    = topic_counts.head(10).index.tolist()

# Bar chart — top 15 topics by size
top15 = topic_model.get_topic_info().iloc[1:16]
plt.figure(figsize=(12, 6))
plt.bar(range(len(top15)), top15['Count'])
plt.xlabel('Topic ID')
plt.ylabel('Number of Tweets')
plt.title('Top 15 Topics by Size')
plt.xticks(range(len(top15)), top15['Topic'], rotation=45)
plt.tight_layout()
plt.savefig(save_path + 'topic_distribution.png', dpi=300)
plt.show()
print("Saved topic_distribution.png")

# BERTopic interactive visualizations
topic_model.visualize_barchart(top_n_topics=10)
topic_model.visualize_topics()

# ─────────────────────────────────────────────
# 10. LDA BASELINE
# ─────────────────────────────────────────────
print("Running LDA baseline...")

vectorizer = CountVectorizer(max_features=5000, max_df=0.95, min_df=2)
dtm = vectorizer.fit_transform(df_sample['clean_text'])

lda = LatentDirichletAllocation(
    n_components=210,
    random_state=42,
    n_jobs=-1
)
lda_topics_matrix = lda.fit_transform(dtm)

lda_labels = lda_topics_matrix.argmax(axis=1)
df_sample['lda_topic'] = lda_labels

lda_dominant = (
    df_sample.groupby('lda_topic')['class']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
lda_dominant.columns = ['lda_topic', 'lda_predicted']
df_sample = df_sample.merge(lda_dominant, on='lda_topic')

print("LDA Sentiment Classification Report:")
print(classification_report(df_sample['class'], df_sample['lda_predicted']))

df_sample.to_csv(save_path + 'df_final.csv', index=False)
print("Saved df_final.csv")

# ─────────────────────────────────────────────
# 11. TRAIN / VAL / TEST SPLIT
# ─────────────────────────────────────────────
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label  = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

df_sample['label'] = df_sample['class'].map(label2id)

train_df, temp_df = train_test_split(
    df_sample,
    test_size=0.2,
    stratify=df_sample['label'],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df['class'].value_counts())

train_df.to_csv(save_path + 'train_df.csv', index=False)
val_df.to_csv(save_path + 'val_df.csv', index=False)
test_df.to_csv(save_path + 'test_df.csv', index=False)
print("Saved train/val/test splits")

# Class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(['Negative', 'Neutral', 'Positive']),
    y=df_sample['class']
)
print(f"Class weights — Negative: {class_weights[0]:.4f} | "
      f"Neutral: {class_weights[1]:.4f} | Positive: {class_weights[2]:.4f}")

weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# ─────────────────────────────────────────────
# 12. SHARED DATASET CLASS & HELPER FUNCTIONS
# ─────────────────────────────────────────────
class ArabicSentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for i, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if i % 200 == 0:
            print(f"  Batch {i}/{len(loader)} — Loss: {loss.item():.4f}")

    return total_loss / len(loader), f1_score(all_labels, all_preds, average='weighted')


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), f1_score(all_labels, all_preds, average='weighted')


def full_evaluation(true_labels, pred_labels, label2id, model_name="Model"):
    true_nums = [label2id[l] for l in true_labels]
    pred_nums = [label2id[p] for p in pred_labels]
    true_bin  = label_binarize(true_nums, classes=[0, 1, 2])
    pred_bin  = label_binarize(pred_nums, classes=[0, 1, 2])

    print(f"\n{'='*55}")
    print(f"{model_name} — Complete Evaluation Metrics")
    print(f"{'='*55}")
    print(classification_report(true_labels, pred_labels))
    print(f"Accuracy:           {accuracy_score(true_labels, pred_labels):.4f}")
    print(f"Macro Precision:    {precision_score(true_labels, pred_labels, average='macro'):.4f}")
    print(f"Macro Recall:       {recall_score(true_labels, pred_labels, average='macro'):.4f}")
    print(f"Macro F1:           {f1_score(true_labels, pred_labels, average='macro'):.4f}")
    print(f"Weighted F1:        {f1_score(true_labels, pred_labels, average='weighted'):.4f}")
    print(f"Cohen Kappa:        {cohen_kappa_score(true_labels, pred_labels):.4f}")
    print(f"Matthews MCC:       {matthews_corrcoef(true_labels, pred_labels):.4f}")
    print(f"ROC AUC (macro):    {roc_auc_score(true_bin, pred_bin, average='macro'):.4f}")
    print(f"{'='*55}")


def plot_confusion_matrix(true_labels, pred_labels, title, save_name):
    cm = confusion_matrix(
        true_labels, pred_labels,
        labels=['Negative', 'Neutral', 'Positive']
    )
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path + save_name, dpi=300)
    plt.show()
    print(f"Saved {save_name}")

# ─────────────────────────────────────────────
# 13. AraBERT SENTIMENT FINE-TUNING
# ─────────────────────────────────────────────
print("\nLoading AraBERT for sentiment classification...")
arabert_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
arabert_model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv2",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)

train_dataset = ArabicSentimentDataset(train_df['clean_text'].tolist(), train_df['label'].tolist(), arabert_tokenizer)
val_dataset   = ArabicSentimentDataset(val_df['clean_text'].tolist(),   val_df['label'].tolist(),   arabert_tokenizer)
test_dataset  = ArabicSentimentDataset(test_df['clean_text'].tolist(),  test_df['label'].tolist(),  arabert_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

loss_fn  = nn.CrossEntropyLoss(weight=weights)
EPOCHS   = 4
patience = 2

optimizer = AdamW(arabert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

best_val_f1, no_improve = 0, 0
print("Starting AraBERT training...")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_f1 = train_epoch(arabert_model, train_loader, optimizer, scheduler, loss_fn, device)
    val_loss, val_f1     = evaluate(arabert_model, val_loader, loss_fn, device)
    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Loss:   {val_loss:.4f}   | Val F1:   {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve  = 0
        torch.save(arabert_model.state_dict(), save_path + 'best_arabert_sentiment.pt')
        print(f"Best model saved (Val F1: {val_f1:.4f})")
    else:
        no_improve += 1
        print(f"No improvement for {no_improve} epoch(s)")
        if no_improve >= patience:
            print("Early stopping triggered")
            break

print(f"\nAraBERT training complete. Best Val F1: {best_val_f1:.4f}")

# AraBERT evaluation
arabert_model.load_state_dict(torch.load(save_path + 'best_arabert_sentiment.pt'))
arabert_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = arabert_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(batch['label'].numpy())

pred_labels_ab = [id2label[p] for p in all_preds]
true_labels_ab = [id2label[l] for l in all_labels]

full_evaluation(true_labels_ab, pred_labels_ab, label2id, model_name="AraBERT")
plot_confusion_matrix(true_labels_ab, pred_labels_ab,
                      "Confusion Matrix — AraBERT", "confusion_matrix_arabert.png")

test_df = test_df.copy()
test_df['arabert_predicted'] = pred_labels_ab
test_df.to_csv(save_path + 'test_results_arabert.csv', index=False)

# ─────────────────────────────────────────────
# 14. MARBERT SENTIMENT FINE-TUNING
# ─────────────────────────────────────────────
print("\nLoading MARBERTv2 for sentiment classification...")
marbert_tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERTv2")
marbert_model = AutoModelForSequenceClassification.from_pretrained(
    "UBC-NLP/MARBERTv2",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)

train_dataset_mb = ArabicSentimentDataset(train_df['clean_text'].tolist(), train_df['label'].tolist(), marbert_tokenizer)
val_dataset_mb   = ArabicSentimentDataset(val_df['clean_text'].tolist(),   val_df['label'].tolist(),   marbert_tokenizer)
test_dataset_mb  = ArabicSentimentDataset(test_df['clean_text'].tolist(),  test_df['label'].tolist(),  marbert_tokenizer)

train_loader_mb = DataLoader(train_dataset_mb, batch_size=32, shuffle=True)
val_loader_mb   = DataLoader(val_dataset_mb,   batch_size=32, shuffle=False)
test_loader_mb  = DataLoader(test_dataset_mb,  batch_size=32, shuffle=False)

loss_fn_mb   = nn.CrossEntropyLoss(weight=weights)
optimizer_mb = AdamW(marbert_model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps_mb = len(train_loader_mb) * EPOCHS
scheduler_mb = get_linear_schedule_with_warmup(
    optimizer_mb,
    num_warmup_steps=total_steps_mb // 10,
    num_training_steps=total_steps_mb
)

best_val_f1_mb, no_improve = 0, 0
print("Starting MARBERT training...")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_f1 = train_epoch(marbert_model, train_loader_mb, optimizer_mb, scheduler_mb, loss_fn_mb, device)
    val_loss, val_f1     = evaluate(marbert_model, val_loader_mb, loss_fn_mb, device)
    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Loss:   {val_loss:.4f}   | Val F1:   {val_f1:.4f}")
    if val_f1 > best_val_f1_mb:
        best_val_f1_mb = val_f1
        no_improve     = 0
        torch.save(marbert_model.state_dict(), save_path + 'best_marbert_sentiment.pt')
        print(f"Best model saved (Val F1: {val_f1:.4f})")
    else:
        no_improve += 1
        print(f"No improvement for {no_improve} epoch(s)")
        if no_improve >= patience:
            print("Early stopping triggered")
            break

print(f"\nMARBERT training complete. Best Val F1: {best_val_f1_mb:.4f}")

# MARBERT evaluation
marbert_model.load_state_dict(torch.load(save_path + 'best_marbert_sentiment.pt'))
marbert_model.eval()

all_preds_mb, all_labels_mb = [], []
with torch.no_grad():
    for batch in test_loader_mb:
        outputs = marbert_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        all_preds_mb.extend(outputs.logits.argmax(dim=1).cpu().numpy())
        all_labels_mb.extend(batch['label'].numpy())

pred_labels_mb = [id2label[p] for p in all_preds_mb]
true_labels_mb = [id2label[l] for l in all_labels_mb]

full_evaluation(true_labels_mb, pred_labels_mb, label2id, model_name="MARBERT")
plot_confusion_matrix(true_labels_mb, pred_labels_mb,
                      "Confusion Matrix — MARBERT", "confusion_matrix_marbert.png")

# Side-by-side comparison print
print(f"\n{'='*55}")
print("FULL COMPARISON: AraBERT vs MARBERT vs LDA")
print(f"{'='*55}")
print(f"{'Metric':<25} {'AraBERT':>10} {'MARBERT':>10} {'LDA':>10}")
print("-" * 55)
metrics_ab = {
    'Accuracy':    accuracy_score(true_labels_ab, pred_labels_ab),
    'Macro F1':    f1_score(true_labels_ab, pred_labels_ab, average='macro'),
    'Weighted F1': f1_score(true_labels_ab, pred_labels_ab, average='weighted'),
}
metrics_mb = {
    'Accuracy':    accuracy_score(true_labels_mb, pred_labels_mb),
    'Macro F1':    f1_score(true_labels_mb, pred_labels_mb, average='macro'),
    'Weighted F1': f1_score(true_labels_mb, pred_labels_mb, average='weighted'),
}
lda_acc = accuracy_score(df_sample['class'], df_sample['lda_predicted'])
lda_mf1 = f1_score(df_sample['class'], df_sample['lda_predicted'], average='macro')
lda_wf1 = f1_score(df_sample['class'], df_sample['lda_predicted'], average='weighted')

for key in metrics_ab:
    lda_val = {'Accuracy': lda_acc, 'Macro F1': lda_mf1, 'Weighted F1': lda_wf1}[key]
    print(f"{key:<25} {metrics_ab[key]:>10.4f} {metrics_mb[key]:>10.4f} {lda_val:>10.4f}")
print("=" * 55)

# ─────────────────────────────────────────────
# 15. MARBERT FULL INFERENCE ON ALL TWEETS
# ─────────────────────────────────────────────
class ArabicInferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts      = texts
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }

full_dataset = ArabicInferenceDataset(df_sample['clean_text'].tolist(), marbert_tokenizer)
full_loader  = DataLoader(full_dataset, batch_size=64, shuffle=False)

print(f"Running MARBERT inference on all {len(df_sample)} tweets...")
all_preds_full = []
marbert_model.eval()
with torch.no_grad():
    for i, batch in enumerate(full_loader):
        outputs = marbert_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        all_preds_full.extend(outputs.logits.argmax(dim=1).cpu().numpy())
        if i % 100 == 0:
            print(f"  Processed {i * 64}/{len(df_sample)} tweets")

df_sample['marbert_sentiment'] = [id2label[p] for p in all_preds_full]
print(f"\nSentiment distribution:")
print(df_sample['marbert_sentiment'].value_counts())

df_sample.to_csv(save_path + 'df_final_marbert.csv', index=False)
print("Saved df_final_marbert.csv")

# ─────────────────────────────────────────────
# 16. SENTIMENT PER TOPIC (MARBERT PREDICTIONS)
# ─────────────────────────────────────────────
def fix_arabic(text):
    return get_display(arabic_reshaper.reshape(text))

sentiment_data = []
for t in top10_ids:
    subset = df_sample[df_sample['topic'] == t]
    total  = len(subset)
    pos = len(subset[subset['marbert_sentiment'] == 'Positive']) / total * 100
    neg = len(subset[subset['marbert_sentiment'] == 'Negative']) / total * 100
    neu = len(subset[subset['marbert_sentiment'] == 'Neutral'])  / total * 100
    dominant = max([('Positive', pos), ('Negative', neg), ('Neutral', neu)], key=lambda x: x[1])[0]
    sentiment_data.append({'topic': t, 'Positive': pos, 'Negative': neg, 'Neutral': neu, 'dominant': dominant})
    print(f"Topic {t}: Neg={neg:.1f}% Neu={neu:.1f}% Pos={pos:.1f}% → {dominant}")

topic_sentiment_df = pd.DataFrame(sentiment_data)
topic_sentiment_df.to_csv(save_path + 'topic_sentiment_marbert.csv', index=False)

# Topic keyword table
topic_rows = []
for t in top10_ids:
    keywords = ", ".join([w for w, s in topic_model.get_topic(t)[:5]])
    row = topic_sentiment_df[topic_sentiment_df['topic'] == t].iloc[0]
    topic_rows.append({
        'Topic ID':          t,
        'Keywords':          keywords,
        'Positive %':        round(row['Positive'], 2),
        'Negative %':        round(row['Negative'], 2),
        'Neutral %':         round(row['Neutral'],  2),
        'Dominant Sentiment': row['dominant']
    })

final_topic_table = pd.DataFrame(topic_rows)
final_topic_table.to_csv(save_path + 'final_topic_sentiment_table.csv', index=False)
print(final_topic_table)

# ─────────────────────────────────────────────
# 17. VISUALIZATIONS
# ─────────────────────────────────────────────

# Overall sentiment bar chart
plt.figure(figsize=(7, 5))
df_sample['marbert_sentiment'].value_counts().plot(kind='bar')
plt.title('Overall Sentiment Distribution — MARBERT')
plt.xlabel('Sentiment')
plt.ylabel('Number of Tweets')
plt.tight_layout()
plt.savefig(save_path + 'overall_sentiment_marbert.png', dpi=300)
plt.show()

# Overall sentiment pie chart
sentiment_counts = df_sample['marbert_sentiment'].value_counts()
plt.figure(figsize=(7, 7))
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Overall Sentiment Distribution — MARBERT')
plt.savefig(save_path + 'sentiment_pie_chart.png', dpi=300)
plt.show()

# Sentiment per topic grouped bar chart
sent_df_plot = []
for t in top10_ids:
    keywords = [fix_arabic(w[0]) for w in topic_model.get_topic(t)[:2]]
    label    = f"Topic {t}\n({', '.join(keywords)})"
    row      = topic_sentiment_df[topic_sentiment_df['topic'] == t].iloc[0]
    sent_df_plot.append({'topic': label, 'Positive': row['Positive'],
                         'Negative': row['Negative'], 'Neutral': row['Neutral']})

sent_df = pd.DataFrame(sent_df_plot)
fig, ax = plt.subplots(figsize=(16, 8))
x, width = range(len(sent_df)), 0.25
ax.bar([i - width for i in x], sent_df['Positive'], width, label='Positive', color='green', alpha=0.7)
ax.bar(x,                       sent_df['Negative'], width, label='Negative', color='red',   alpha=0.7)
ax.bar([i + width for i in x], sent_df['Neutral'],  width, label='Neutral',  color='gray',  alpha=0.7)
ax.set_xlabel('Topic')
ax.set_ylabel('Percentage (%)')
ax.set_title('Sentiment Distribution per Topic — MARBERT')
ax.set_xticks(x)
ax.set_xticklabels(sent_df['topic'], rotation=45, ha='right', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig(save_path + 'sentiment_per_topic_marbert.png', dpi=300)
plt.show()

# ─────────────────────────────────────────────
# 18. SAVE ALL FINAL OUTPUTS
# ─────────────────────────────────────────────
final_topic_table.to_csv(save_path + 'final_topic_table.csv', index=False)
topic_sentiment_df.to_csv(save_path + 'topic_sentiment_summary.csv', index=False)
df_sample.to_csv(save_path + 'final_dataset_with_predictions.csv', index=False)
print("All outputs saved successfully.")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 28.5 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install dependencies
# ─────────────────────────────────────────────
!pip install bertopic transformers sentence-transformers umap-learn hdbscan arabert pyarabic arabic-reshaper python-bidi -q

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Imports
# ─────────────────────────────────────────────
import os
import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import pyarabic.araby as araby
import arabic_reshaper
from bidi.algorithm import get_display
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    cohen_kappa_score, matthews_corrcoef, roc_auc_score,
)
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
print("Imports done")

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Mount Drive & set path
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/arabic_opinion_clustering/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Load saved dataframes & embeddings
# ─────────────────────────────────────────────
df_sample  = pd.read_csv(save_path + 'df_final_marbert.csv')
train_df   = pd.read_csv(save_path + 'train_df.csv')
val_df     = pd.read_csv(save_path + 'val_df.csv')
test_df    = pd.read_csv(save_path + 'test_df.csv')
embeddings = np.load(save_path + 'embeddings.npy')

print(f"df_sample:  {df_sample.shape}")
print(f"train_df:   {train_df.shape}")
print(f"embeddings: {embeddings.shape}")
print(f"Columns: {df_sample.columns.tolist()}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Load BERTopic model
# ─────────────────────────────────────────────
topic_model = BERTopic.load(save_path + 'bertopic_model')
print("BERTopic loaded")
print(f"Number of topics: {len(topic_model.get_topic_info())}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Load MARBERT model
# ─────────────────────────────────────────────
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label  = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

marbert_tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERTv2")
marbert_model = AutoModelForSequenceClassification.from_pretrained(
    "UBC-NLP/MARBERTv2",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)
marbert_model.load_state_dict(torch.load(save_path + 'best_marbert_sentiment.pt', map_location=device))
marbert_model.eval()
print("MARBERT loaded")

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Load AraBERT model
# ─────────────────────────────────────────────
arabert_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
arabert_model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv2",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)
arabert_model.load_state_dict(torch.load(save_path + 'best_arabert_sentiment.pt', map_location=device))
arabert_model.eval()
print("AraBERT loaded")

In [ ]:

# ─────────────────────────────────────────────
# CELL 8 — Verify state + fix random seeds
# ─────────────────────────────────────────────
import random
import numpy as np
import torch

# Set all random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
print("Random seeds set ✓")

# Check df_sample state
print(f"\ndf_sample shape:  {df_sample.shape}")
print(f"Columns: {df_sample.columns.tolist()}")

# Check topic distribution
if 'topic' in df_sample.columns:
    n_topics = df_sample['topic'].nunique()
    n_noise  = (df_sample['topic'] == -1).sum()
    print(f"\nTopics: {n_topics} | Noise: {n_noise}")
    print(df_sample['topic'].value_counts().head(10))

# Remap labels (lost after CSV reload)
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df_sample['label'] = df_sample['class'].map(label2id)
train_df['label']  = train_df['class'].map(label2id)
val_df['label']    = val_df['class'].map(label2id)
test_df['label']   = test_df['class'].map(label2id)
print("\nLabels remapped ✓")

# Redefine top topics
topic_counts = df_sample[df_sample['topic'] != -1]['topic'].value_counts()
top10_ids = topic_counts.head(10).index.tolist()
print(f"Top 10 topics: {top10_ids}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Arabic stop words + update keywords
# ─────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

arabic_stopwords = [
    # Pronouns
    'انا', 'انت', 'انتي', 'انتم', 'انتن', 'هو', 'هي', 'هم', 'هن', 'نحن',
    # Prepositions & conjunctions
    'في', 'من', 'على', 'إلى', 'الى', 'عن', 'مع', 'بعد', 'قبل', 'حتى',
    'لكن', 'او', 'أو', 'إن', 'ان', 'اذا', 'اذ', 'لو', 'كي', 'ثم',
    # Common words
    'كل', 'هذا', 'هذه', 'ذلك', 'تلك', 'الذي', 'التي', 'الذين',
    'كان', 'كانت', 'يكون', 'تكون', 'ليس', 'ليست',
    'قال', 'قالت', 'يقول', 'تقول',
    'اي', 'أي', 'اين', 'أين', 'كيف', 'لماذا', 'متى', 'ما', 'ماذا',
    'لا', 'لم', 'لن', 'قد', 'سوف', 'سي',
    'عند', 'عندما', 'حين', 'بين', 'خلال', 'حول', 'تحت', 'فوق',
    # Tweet noise
    'الي', 'اشي', 'انه', 'انها', 'فيه', 'فيها', 'عليه', 'عليها',
    'منه', 'منها', 'بها', 'به', 'لها', 'له', 'عنه', 'عنها',
    'هل', 'نعم', 'اه', 'اوه', 'وش', 'شو',
    'يا', 'يلا', 'بس', 'كده', 'كدا', 'زي', 'يعني',
    # Articles
    'ال', 'و', 'ف', 'ب', 'ك', 'ل',
]

vectorizer_model = CountVectorizer(
    stop_words=arabic_stopwords,
    min_df=5,
    ngram_range=(1, 2)
)

docs = df_sample['clean_text'].tolist()
print("Updating topic keywords with stop words...")
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)

# Save updated model
topic_model.save(save_path + 'bertopic_model_cleaned')
print("Saved bertopic_model_cleaned ✓")

# Show top 5 topics
print("\nTop 5 topics after stop word removal:")
for i, tid in enumerate(top10_ids[:5]):
    kws = [w for w, s in topic_model.get_topic(tid)[:5]]
    print(f"  {i+1}. Topic {tid}: {kws}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — LDA baseline (CORRECTED)
# ─────────────────────────────────────────────
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

# Drop old LDA columns to avoid conflicts
df_sample = df_sample.drop(
    columns=['lda_topic', 'lda_predicted', 'lda_predicted_x', 'lda_predicted_y'],
    errors='ignore'
)
print("Dropped old LDA columns ✓")

# Vectorize
vectorizer_lda = CountVectorizer(max_features=5000, max_df=0.95, min_df=2)
dtm = vectorizer_lda.fit_transform(df_sample['clean_text'])

# Fit LDA
lda = LatentDirichletAllocation(
    n_components=210,
    random_state=42,
    n_jobs=-1
)
lda_topics_matrix = lda.fit_transform(dtm)

df_sample['lda_topic'] = lda_topics_matrix.argmax(axis=1)

# ✅ Recreate same train/test split indices using same seed
train_idx, temp_idx = train_test_split(
    df_sample.index,
    test_size=0.2,
    stratify=df_sample['label'],
    random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=df_sample.loc[temp_idx, 'label'],
    random_state=42
)
print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# ✅ Get dominant sentiment per topic using TRAIN rows only
lda_dominant = (
    df_sample.loc[train_idx]
    .groupby('lda_topic')['class']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
lda_dominant.columns = ['lda_topic', 'lda_predicted']

# Merge back
df_sample = df_sample.merge(lda_dominant, on='lda_topic', how='left')
df_sample['lda_predicted'] = df_sample['lda_predicted'].fillna('Neutral')

# ✅ Evaluate on TEST rows only
lda_test_true = df_sample.loc[test_idx, 'class']
lda_test_pred = df_sample.loc[test_idx, 'lda_predicted']

print("\n✅ LDA Baseline — Test Set Only:")
print(classification_report(lda_test_true, lda_test_pred))

lda_acc = accuracy_score(lda_test_true, lda_test_pred)
lda_f1_macro = f1_score(lda_test_true, lda_test_pred, average='macro')
lda_f1_weighted = f1_score(lda_test_true, lda_test_pred, average='weighted')

print(f"Accuracy:    {lda_acc:.4f}")
print(f"Macro F1:    {lda_f1_macro:.4f}")
print(f"Weighted F1: {lda_f1_weighted:.4f}")

# Save
df_sample.to_csv(save_path + 'df_with_lda_corrected.csv', index=False)
print("\nSaved df_with_lda_corrected.csv ✓")

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — NPMI Coherence (Fast Version)
# ─────────────────────────────────────────────
!pip install gensim -q
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# Sample 20k docs — sufficient for valid coherence score
sample_docs = df_sample['clean_text'].sample(20000, random_state=42).tolist()
texts_tokenized = [doc.split() for doc in sample_docs]
dictionary = Dictionary(texts_tokenized)
print(f"Dictionary size: {len(dictionary)} tokens")

# BERTopic top words per topic (top 30 topics only)
bertopic_topics = []
valid_topic_ids = [t for t in topic_model.get_topic_info()['Topic'] if t != -1][:30]
for tid in valid_topic_ids:
    words = [w for w, s in topic_model.get_topic(tid)[:10]
             if w in dictionary.token2id]
    if len(words) >= 3:
        bertopic_topics.append(words)
print(f"BERTopic topics for coherence: {len(bertopic_topics)}")

# LDA top words per topic (top 30 topics only)
lda_topics_words = []
feature_names = vectorizer_lda.get_feature_names_out()
for topic_vec in lda.components_[:30]:
    top_indices = topic_vec.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_indices
             if feature_names[i] in dictionary.token2id]
    if len(words) >= 3:
        lda_topics_words.append(words)
print(f"LDA topics for coherence: {len(lda_topics_words)}")

# BERTopic coherence
print("\nCalculating BERTopic NPMI...")
cm_bertopic = CoherenceModel(
    topics=bertopic_topics,
    texts=texts_tokenized,
    dictionary=dictionary,
    coherence='c_npmi'
)
bertopic_coherence = cm_bertopic.get_coherence()
print(f"BERTopic NPMI: {bertopic_coherence:.4f}")

# LDA coherence
print("Calculating LDA NPMI...")
cm_lda = CoherenceModel(
    topics=lda_topics_words,
    texts=texts_tokenized,
    dictionary=dictionary,
    coherence='c_npmi'
)
lda_coherence = cm_lda.get_coherence()
print(f"LDA NPMI: {lda_coherence:.4f}")

print(f"\n{'='*40}")
print(f"BERTopic NPMI: {bertopic_coherence:.4f}")
print(f"LDA NPMI:      {lda_coherence:.4f}")
print(f"Improvement:   {bertopic_coherence - lda_coherence:+.4f}")
print(f"{'='*40}")

# Save
pd.DataFrame({
    'Model': ['BERTopic', 'LDA'],
    'NPMI_Coherence': [bertopic_coherence, lda_coherence]
}).to_csv(save_path + 'coherence_scores.csv', index=False)
print("Saved coherence_scores.csv ✓")

In [ ]:
!pip install statsmodels -q

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — AraBERT results (load from saved file)
# ─────────────────────────────────────────────
test_results = pd.read_csv(save_path + 'test_results.csv')
print(f"Loaded test_results.csv: {test_results.shape}")
print(f"Columns: {test_results.columns.tolist()}")
print(test_results.head(3))

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — McNemar's Test (MARBERT vs AraBERT)
# ─────────────────────────────────────────────
from statsmodels.stats.contingency_tables import mcnemar

# Create contingency table
marbert_correct = np.array(pred_labels_mb) == np.array(true_labels_mb)
arabert_correct = np.array(pred_labels_ab) == np.array(true_labels_ab)

both_correct = np.sum(marbert_correct & arabert_correct)
marbert_only = np.sum(marbert_correct & ~arabert_correct)
arabert_only = np.sum(~marbert_correct & arabert_correct)
both_wrong   = np.sum(~marbert_correct & ~arabert_correct)

contingency_table = np.array([[both_correct, arabert_only],
                               [marbert_only, both_wrong]])

print(f"\n{'='*50}")
print("McNemar's Test: MARBERT vs AraBERT")
print(f"{'='*50}")
print(f"Both correct:     {both_correct}")
print(f"MARBERT only:     {marbert_only}")
print(f"AraBERT only:     {arabert_only}")
print(f"Both wrong:       {both_wrong}")

result = mcnemar(contingency_table, exact=False, correction=True)
print(f"\nMcNemar statistic: {result.statistic:.4f}")
print(f"P-value:           {result.pvalue:.4f}")

if result.pvalue < 0.05:
    print("✅ MARBERT is SIGNIFICANTLY better than AraBERT (p < 0.05)")
else:
    print("❌ No significant difference between models (p >= 0.05)")
print(f"{'='*50}")

# Save results
mcnemar_results = pd.DataFrame({
    'Metric': ['Both Correct', 'MARBERT Only', 'AraBERT Only', 'Both Wrong',
               'McNemar Statistic', 'P-value'],
    'Value': [both_correct, marbert_only, arabert_only, both_wrong,
              result.statistic, result.pvalue]
})
mcnemar_results.to_csv(save_path + 'mcnemar_test_results.csv', index=False)
print("Saved mcnemar_test_results.csv ✓")

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 RECOVERY — Load MARBERT results
# ─────────────────────────────────────────────
df_final_marbert = pd.read_csv(save_path + 'df_final_marbert.csv')
print(f"Columns: {df_final_marbert.columns.tolist()}")
print(df_final_marbert.head(3))

In [ ]:
# ─────────────────────────────────────────────
# CELL 12+13 RECOVERY (FIXED) — Correct MARBERT test predictions
# ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split

# AraBERT is fine — keep as is
true_labels_ab = test_results['class'].tolist()
pred_labels_ab = test_results['arabert_predicted'].tolist()
print(f"AraBERT — {len(true_labels_ab)} samples ✓")

# MARBERT — recreate same split on df_final_marbert using same seed
df_final_marbert['label'] = df_final_marbert['class'].map(label2id)

_, temp_idx = train_test_split(
    df_final_marbert.index,
    test_size=0.2,
    stratify=df_final_marbert['label'],
    random_state=42
)
_, test_idx_mb = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=df_final_marbert.loc[temp_idx, 'label'],
    random_state=42
)

test_mb = df_final_marbert.loc[test_idx_mb]
true_labels_mb = test_mb['class'].tolist()
pred_labels_mb = test_mb['marbert_sentiment'].tolist()

print(f"MARBERT — {len(true_labels_mb)} samples ✓")
print(f"\nClass distribution check:")
print(pd.Series(true_labels_mb).value_counts())
print(f"\nSample: true={true_labels_mb[:3]} | pred={pred_labels_mb[:3]}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — McNemar's Test
# ─────────────────────────────────────────────
!pip install statsmodels -q
from statsmodels.stats.contingency_tables import mcnemar

marbert_correct = np.array(pred_labels_mb) == np.array(true_labels_mb)
arabert_correct = np.array(pred_labels_ab) == np.array(true_labels_ab)

both_correct = np.sum(marbert_correct & arabert_correct)
marbert_only = np.sum(marbert_correct & ~arabert_correct)
arabert_only = np.sum(~marbert_correct & arabert_correct)
both_wrong   = np.sum(~marbert_correct & ~arabert_correct)

contingency_table = np.array([[both_correct, arabert_only],
                               [marbert_only, both_wrong]])

result = mcnemar(contingency_table, exact=False, correction=True)

print(f"{'='*50}")
print("McNemar's Test: MARBERT vs AraBERT")
print(f"{'='*50}")
print(f"Both correct:     {both_correct}")
print(f"MARBERT only:     {marbert_only}")
print(f"AraBERT only:     {arabert_only}")
print(f"Both wrong:       {both_wrong}")
print(f"\nMcNemar statistic: {result.statistic:.4f}")
print(f"P-value:           {result.pvalue:.6f}")
if result.pvalue < 0.05:
    print("✅ MARBERT is SIGNIFICANTLY better than AraBERT (p < 0.05)")
else:
    print("❌ No significant difference found (p >= 0.05)")
print(f"{'='*50}")

pd.DataFrame({
    'Metric': ['Both Correct', 'MARBERT Only', 'AraBERT Only',
               'Both Wrong', 'McNemar Statistic', 'P-value'],
    'Value':  [both_correct, marbert_only, arabert_only,
               both_wrong, result.statistic, result.pvalue]
}).to_csv(save_path + 'mcnemar_results.csv', index=False)
print("Saved mcnemar_results.csv ✓")

In [ ]:
# ─────────────────────────────────────────────
# CELL 15 — Confusion Matrices (Side by Side)
# ─────────────────────────────────────────────
from sklearn.metrics import confusion_matrix
import seaborn as sns

labels = ['Negative', 'Neutral', 'Positive']
cm_ab = confusion_matrix(true_labels_ab, pred_labels_ab, labels=labels)
cm_mb = confusion_matrix(true_labels_mb, pred_labels_mb, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_ab, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix — AraBERT')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_mb, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Confusion Matrix — MARBERT')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(save_path + 'confusion_matrices_side_by_side.png', dpi=300)
plt.show()
print("Saved confusion_matrices_side_by_side.png ✓")

In [ ]:
# ─────────────────────────────────────────────
# CELL 16 — Sentiment Uniformity Per Cluster
# ─────────────────────────────────────────────

# BERTopic uniformity
bertopic_uniformity = []
for tid in df_sample[df_sample['topic'] != -1]['topic'].unique():
    subset = df_sample[df_sample['topic'] == tid]
    dominant = subset['marbert_sentiment'].value_counts().index[0]
    uniformity = (subset['marbert_sentiment'] == dominant).sum() / len(subset)
    bertopic_uniformity.append(uniformity)

# LDA uniformity
lda_uniformity = []
for tid in df_sample['lda_topic'].unique():
    subset = df_sample[df_sample['lda_topic'] == tid]
    dominant = subset['marbert_sentiment'].value_counts().index[0]
    uniformity = (subset['marbert_sentiment'] == dominant).sum() / len(subset)
    lda_uniformity.append(uniformity)

bert_uni_mean = np.mean(bertopic_uniformity)
lda_uni_mean  = np.mean(lda_uniformity)

print(f"{'='*50}")
print("Sentiment Uniformity Per Cluster")
print(f"{'='*50}")
print(f"BERTopic mean uniformity: {bert_uni_mean:.4f}")
print(f"LDA mean uniformity:      {lda_uni_mean:.4f}")
print(f"Improvement:              {bert_uni_mean - lda_uni_mean:+.4f}")
print(f"{'='*50}")

pd.DataFrame({
    'Model': ['BERTopic', 'LDA'],
    'Mean_Uniformity': [bert_uni_mean, lda_uni_mean]
}).to_csv(save_path + 'sentiment_uniformity.csv', index=False)
print("Saved sentiment_uniformity.csv ✓")

In [ ]:
# ─────────────────────────────────────────────
# CELL 17 — Final Results Table
# ─────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, cohen_kappa_score, matthews_corrcoef,
                             roc_auc_score)
from sklearn.preprocessing import label_binarize

def get_all_metrics(true, pred, label2id):
    true_nums = [label2id[l] for l in true]
    pred_nums = [label2id[p] for p in pred]
    true_bin  = label_binarize(true_nums, classes=[0,1,2])
    pred_bin  = label_binarize(pred_nums, classes=[0,1,2])
    return {
        'Accuracy':          round(accuracy_score(true, pred), 4),
        'Macro Precision':   round(precision_score(true, pred, average='macro'), 4),
        'Macro Recall':      round(recall_score(true, pred, average='macro'), 4),
        'Macro F1':          round(f1_score(true, pred, average='macro'), 4),
        'Weighted F1':       round(f1_score(true, pred, average='weighted'), 4),
        'Cohen Kappa':       round(cohen_kappa_score(true, pred), 4),
        'Matthews MCC':      round(matthews_corrcoef(true, pred), 4),
        'ROC AUC':           round(roc_auc_score(true_bin, pred_bin, average='macro'), 4),
    }

# Load coherence and uniformity
coherence_df  = pd.read_csv(save_path + 'coherence_scores.csv')
uniformity_df = pd.read_csv(save_path + 'sentiment_uniformity.csv')

bert_coherence = coherence_df[coherence_df['Model'] == 'BERTopic']['NPMI_Coherence'].values[0]
lda_coherence  = coherence_df[coherence_df['Model'] == 'LDA']['NPMI_Coherence'].values[0]
bert_uni       = uniformity_df[uniformity_df['Model'] == 'BERTopic']['Mean_Uniformity'].values[0]
lda_uni        = uniformity_df[uniformity_df['Model'] == 'LDA']['Mean_Uniformity'].values[0]

# Get metrics
metrics_ab  = get_all_metrics(true_labels_ab, pred_labels_ab, label2id)
metrics_mb  = get_all_metrics(true_labels_mb, pred_labels_mb, label2id)

lda_test_true = df_sample.loc[test_idx, 'class']
lda_test_pred = df_sample.loc[test_idx, 'lda_predicted']
metrics_lda = {
    'Accuracy':        round(accuracy_score(lda_test_true, lda_test_pred), 4),
    'Macro Precision': round(precision_score(lda_test_true, lda_test_pred, average='macro'), 4),
    'Macro Recall':    round(recall_score(lda_test_true, lda_test_pred, average='macro'), 4),
    'Macro F1':        round(f1_score(lda_test_true, lda_test_pred, average='macro'), 4),
    'Weighted F1':     round(f1_score(lda_test_true, lda_test_pred, average='weighted'), 4),
    'Cohen Kappa':     round(cohen_kappa_score(lda_test_true, lda_test_pred), 4),
    'Matthews MCC':    round(matthews_corrcoef(lda_test_true, lda_test_pred), 4),
    'ROC AUC':         'N/A',
}

# Build table
results_df = pd.DataFrame({
    'Metric':             list(metrics_ab.keys()) + ['NPMI Coherence', 'Sentiment Uniformity'],
    'AraBERT':            list(metrics_ab.values()) + ['N/A', 'N/A'],
    'MARBERT':            list(metrics_mb.values()) + [round(bert_coherence, 4), round(bert_uni, 4)],
    'LDA (Baseline)':     list(metrics_lda.values()) + [round(lda_coherence, 4), round(lda_uni, 4)],
})

print(f"\n{'='*70}")
print("FINAL RESULTS TABLE")
print(f"{'='*70}")
print(results_df.to_string(index=False))
print(f"{'='*70}")

results_df.to_csv(save_path + 'final_results_table.csv', index=False)
print("\nSaved final_results_table.csv ✓")

In [ ]:
# ─────────────────────────────────────────────
# CELL 18 — Qualitative Topic Analysis
# ─────────────────────────────────────────────
topic_rows = []
topic_counts = df_sample[df_sample['topic'] != -1]['topic'].value_counts()
top10_ids = topic_counts.head(10).index.tolist()

for tid in top10_ids:
    subset  = df_sample[df_sample['topic'] == tid]
    total   = len(subset)
    pos = len(subset[subset['marbert_sentiment'] == 'Positive']) / total * 100
    neg = len(subset[subset['marbert_sentiment'] == 'Negative']) / total * 100
    neu = len(subset[subset['marbert_sentiment'] == 'Neutral'])  / total * 100
    dominant = max([('Positive',pos),('Negative',neg),('Neutral',neu)],
                    key=lambda x: x[1])[0]
    try:
        keywords = ', '.join([w for w, s in topic_model.get_topic(tid)[:7]])
    except:
        keywords = 'N/A'

    topic_rows.append({
        'Topic ID':           tid,
        'Size':               total,
        'Keywords':           keywords,
        'Negative %':         round(neg, 1),
        'Neutral %':          round(neu, 1),
        'Positive %':         round(pos, 1),
        'Dominant Sentiment': dominant
    })
    print(f"\nTopic {tid} ({total} tweets) — {dominant}")
    print(f"  Keywords: {keywords}")
    print(f"  Neg: {neg:.1f}% | Neu: {neu:.1f}% | Pos: {pos:.1f}%")

qualitative_df = pd.DataFrame(topic_rows)
qualitative_df.to_csv(save_path + 'qualitative_topic_analysis.csv', index=False)
print("\nSaved qualitative_topic_analysis.csv ✓")

In [ ]:
# ─────────────────────────────────────────────
# CELL 19 — Final save of all outputs
# ─────────────────────────────────────────────
df_sample.to_csv(save_path + 'final_dataset_complete.csv', index=False)

print("All files saved:")
for f in sorted(os.listdir(save_path)):
    size = os.path.getsize(os.path.join(save_path, f)) / 1e6
    print(f"  {f:<50} {size:.1f} MB")

In [ ]:
# ─────────────────────────────────────────────
# CLEANUP — Delete redundant files
# ─────────────────────────────────────────────
import os

files_to_delete = [
    # Old experiment artifacts
    'cluster_labels.npy',        # from dead standalone HDBSCAN
    'df_clustered.csv',          # from dead standalone HDBSCAN
    'df_balanced.csv',           # never used
    'df_evaluation.csv',         # superseded by final_results_table
    'df_final.csv',              # superseded by df_with_lda_corrected
    'df_sample.csv',             # superseded by final_dataset_complete
    'df_topics.csv',             # superseded by df_topics_reduced
    'model_comparison.png',      # old hardcoded chart
    'sentiment_per_topic.png',   # old version
    'final_topic_sentiment_table.csv',  # empty
    'final_topic_table.csv',            # empty
    'topic_sentiment_marbert.csv',      # empty
    'topic_sentiment_summary.csv',      # empty
    # Old BERTopic versions (keep only cleaned one)
    'bertopic_model_25topics',   # we decided against 25 topics
    # Raw/redundant splits
    'test_clean.csv',
    'test_raw.csv',
    'train_clean.csv',
    'train_raw.csv',
    'val_clean.csv',
    'val_raw.csv',
    'train_embeddings.npy',
    'train_reduced_embeddings.npy',
    'umap_model.pkl',
]

for f in files_to_delete:
    path = save_path + f
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {f}")
    else:
        print(f"Not found: {f}")

In [ ]:
# ─────────────────────────────────────────────
# TABLE 1 — Dataset Statistics
# ─────────────────────────────────────────────
train_dist = train_df['class'].value_counts()
val_dist   = val_df['class'].value_counts()
test_dist  = test_df['class'].value_counts()
full_dist  = df_sample['class'].value_counts()

table1 = pd.DataFrame({
    'Split':    ['Train', 'Validation', 'Test', 'Total'],
    'Negative': [train_dist.get('Negative',0), val_dist.get('Negative',0),
                 test_dist.get('Negative',0),  full_dist.get('Negative',0)],
    'Neutral':  [train_dist.get('Neutral',0),  val_dist.get('Neutral',0),
                 test_dist.get('Neutral',0),   full_dist.get('Neutral',0)],
    'Positive': [train_dist.get('Positive',0), val_dist.get('Positive',0),
                 test_dist.get('Positive',0),  full_dist.get('Positive',0)],
    'Total':    [len(train_df), len(val_df), len(test_df), len(df_sample)]
})

print("TABLE 1 — Dataset Statistics")
print(table1.to_string(index=False))
table1.to_csv(save_path + 'table1_dataset_stats.csv', index=False)
print("Saved table1_dataset_stats.csv ✓")

In [ ]:
# ─────────────────────────────────────────────
# TABLE 2 — Sentiment Classification Results
# ─────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, cohen_kappa_score, matthews_corrcoef,
                             roc_auc_score)
from sklearn.preprocessing import label_binarize

def get_metrics(true, pred, label2id, include_roc=True):
    true_nums = [label2id[l] for l in true]
    pred_nums = [label2id[p] for p in pred]
    true_bin  = label_binarize(true_nums, classes=[0,1,2])
    pred_bin  = label_binarize(pred_nums, classes=[0,1,2])
    metrics = {
        'Accuracy':        round(accuracy_score(true, pred), 4),
        'Macro Precision': round(precision_score(true, pred, average='macro', zero_division=0), 4),
        'Macro Recall':    round(recall_score(true, pred, average='macro', zero_division=0), 4),
        'Macro F1':        round(f1_score(true, pred, average='macro', zero_division=0), 4),
        'Weighted F1':     round(f1_score(true, pred, average='weighted', zero_division=0), 4),
        'Cohen Kappa':     round(cohen_kappa_score(true, pred), 4),
        'Matthews MCC':    round(matthews_corrcoef(true, pred), 4),
        'ROC AUC':         round(roc_auc_score(true_bin, pred_bin, average='macro'), 4)
                           if include_roc else 'N/A',
    }
    return metrics

metrics_ab  = get_metrics(true_labels_ab, pred_labels_ab, label2id)
metrics_mb  = get_metrics(true_labels_mb, pred_labels_mb, label2id)

lda_true = df_sample.loc[test_idx, 'class'].tolist()
lda_pred = df_sample.loc[test_idx, 'lda_predicted'].tolist()
metrics_lda = get_metrics(lda_true, lda_pred, label2id, include_roc=False)

table2 = pd.DataFrame({
    'Metric':        list(metrics_ab.keys()),
    'AraBERT':       list(metrics_ab.values()),
    'MARBERT':       list(metrics_mb.values()),
    'Baseline':      list(metrics_lda.values()),
})

print("\nTABLE 2 — Sentiment Classification Results")
print(table2.to_string(index=False))
table2.to_csv(save_path + 'table2_classification_results.csv', index=False)
print("Saved table2_classification_results.csv ✓")

In [ ]:
  import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

save_path = '/content/drive/MyDrive/arabic_opinion_clustering/'
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label  = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

# Load necessary dataframes to recreate labels
test_results = pd.read_csv(save_path + 'test_results.csv')
df_final_marbert = pd.read_csv(save_path + 'df_final_marbert.csv')

# AraBERT labels
true_labels_ab = test_results['class'].tolist()
pred_labels_ab = test_results['arabert_predicted'].tolist()

# MARBERT labels - recreate the test_idx for correct slicing
df_final_marbert['label'] = df_final_marbert['class'].map(label2id)
_, temp_idx = train_test_split(
    df_final_marbert.index,
    test_size=0.2,
    stratify=df_final_marbert['label'],
    random_state=42
)
_, test_idx_mb = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=df_final_marbert.loc[temp_idx, 'label'],
    random_state=42
)
test_mb = df_final_marbert.loc[test_idx_mb]
true_labels_mb = test_mb['class'].tolist()
pred_labels_mb = test_mb['marbert_sentiment'].tolist()

# LDA labels - using the same test_idx_mb for consistency
lda_true = df_final_marbert.loc[test_idx_mb, 'class'].tolist()
lda_pred = df_final_marbert.loc[test_idx_mb, 'lda_predicted'].tolist()

# ─────────────────────────────────────────────
# GRAPH 5 (UPDATED) — 3 Confusion Matrices
# ─────────────────────────────────────────────

labels_order = ['Negative', 'Neutral', 'Positive']
cm_ab  = confusion_matrix(true_labels_ab, pred_labels_ab, labels=labels_order)
cm_mb  = confusion_matrix(true_labels_mb, pred_labels_mb, labels=labels_order)
cm_lda = confusion_matrix(lda_true, lda_pred, labels=labels_order)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, cm, title, cmap in zip(
    axes,
    [cm_ab,    cm_mb, cm_lda],
    ['AraBERT', 'MARBERT', 'LDA'],
    ['Blues',   'Greens', ] # Added cmap for LDA
):
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    sns.heatmap(cm, annot=False, cmap=cmap,
                xticklabels=labels_order,
                yticklabels=labels_order,
                ax=ax, linewidths=0.5,
                cbar_kws={'shrink': 0.8})

    for i in range(len(labels_order)):
        for j in range(len(labels_order)):
            ax.text(j + 0.5, i + 0.5,
                    f'{cm[i,j]:,}\n({cm_norm[i,j]*100:.1f}%)',
                    ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    color='white' if cm_norm[i,j] > 0.5 else 'black')

    acc = np.trace(cm) / np.sum(cm)
    ax.set_title(f'{title}\nAccuracy: {acc:.4f}',
                 fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)

plt.suptitle('Confusion Matrices — AraBERT vs MARBERT vs LDA (Test Set, n=9,974)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(save_path + 'graph5_confusion_matrices_all3.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved graph5_confusion_matrices_all3.png ✓")

In [ ]:
# ─────────────────────────────────────────────
# GRAPH 2 (FIXED) — No collision version
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 8), sharey=True)

for ax, sentiment, skey in zip(
    axes,
    ['Negative', 'Neutral', 'Positive'],
    ['neg',      'neu',     'pos']
):
    for model, offset in model_offsets.items():
        bars = ax.bar(
            x + offset,
            results[model][skey],
            width,
            label=model,
            color=model_colors[model],
            alpha=0.85,
            edgecolor='white',
            linewidth=1.2
        )
        for bar in bars:
            h = bar.get_height()
            if h > 5:
                ax.text(bar.get_x() + bar.get_width()/2,
                        h + 0.8,
                        f'{h:.1f}%',
                        ha='center', va='bottom',
                        fontsize=8, fontweight='bold')

    # ✅ FIX: Put n= inside plot at top instead of below axis
    for i, size in enumerate(sizes):
        ax.text(i, 103,
                f'n={size}',
                ha='center', va='bottom',
                fontsize=7.5, color='dimgray',
                style='italic',
                bbox=dict(boxstyle='round,pad=0.2',
                         facecolor='white',
                         edgecolor='lightgray',
                         alpha=0.8))

    ax.set_title(f'{sentiment} Sentiment %',
                 fontsize=12, fontweight='bold',
                 color=colors[sentiment])

    # ✅ FIX: Rotate labels and add padding
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9,
                       rotation=20, ha='right')
    ax.set_ylim(0, 115)
    ax.yaxis.grid(True, alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if ax == axes[0]:
        ax.set_ylabel('Percentage (%)', fontsize=11)
    ax.legend(fontsize=9, loc='upper left')

plt.suptitle(
    'Sentiment Distribution per Topic — MARBERT vs AraBERT vs LDA\n'
    f'(Test Set Only, n=9,974)',
    fontsize=14, fontweight='bold', y=1.02
)

# ✅ FIX: Add bottom margin for rotated labels
plt.tight_layout()
plt.subplots_adjust(bottom=0.18)

plt.savefig(save_path + 'graph2_sentiment_per_topic_all3.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Saved graph2_sentiment_per_topic_all3.png ✓")

In [ ]:
# ─────────────────────────────────────────────
# GRAPH — Accuracy Comparison (Fixed)
# ─────────────────────────────────────────────
models     = ['LDA\nBaseline', 'AraBERT', 'MARBERT']
accuracies = [
    metrics_lda['Accuracy'],
    metrics_ab['Accuracy'],
    metrics_mb['Accuracy']
]
bar_colors = ['#E67E22', '#3498DB', '#2ECC71']

fig, ax = plt.subplots(figsize=(9, 7))

bars = ax.bar(models, accuracies, color=bar_colors,
              width=0.45, edgecolor='white', linewidth=1.5)

# Value labels on top of bars
for bar, val in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.008,
            f'{val:.4f}',
            ha='center', va='bottom',
            fontsize=13, fontweight='bold')



ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy Comparison — LDA vs AraBERT vs MARBERT\n'
             '(Test Set, n=9,974)', fontsize=13, fontweight='bold')

# ✅ FIX: Extend ylim to give room above bars
ax.set_ylim(0, 1.05)
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(save_path + 'graph_accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved graph_accuracy_comparison.png ✓")

In [ ]:
# ─────────────────────────────────────────────
# GRAPH — F1 Score Comparison (Macro & Weighted)
# ─────────────────────────────────────────────
models = ['LDA\nBaseline', 'AraBERT', 'MARBERT']

macro_f1    = [metrics_lda['Macro F1'],    metrics_ab['Macro F1'],    metrics_mb['Macro F1']]
weighted_f1 = [metrics_lda['Weighted F1'], metrics_ab['Weighted F1'], metrics_mb['Weighted F1']]

x     = np.arange(len(models))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 7))

bars1 = ax.bar(x - width/2, macro_f1,    width,
               label='Macro F1',    color=['#E67E22', '#3498DB', '#2ECC71'],
               alpha=0.85, edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, weighted_f1, width,
               label='Weighted F1', color=['#E67E22', '#3498DB', '#2ECC71'],
               alpha=0.5,  edgecolor='white', linewidth=1.5, hatch='//')

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.008,
            f'{bar.get_height():.4f}',
            ha='center', va='bottom',
            fontsize=10, fontweight='bold')

for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.008,
            f'{bar.get_height():.4f}',
            ha='center', va='bottom',
            fontsize=10, fontweight='bold')



ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('F1 Score Comparison — LDA vs AraBERT vs MARBERT\n'
             '(Test Set, n=9,974)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=12)
ax.set_ylim(0, 1.05)
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=11, loc='lower right')

plt.tight_layout()
plt.savefig(save_path + 'graph_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved graph_f1_comparison.png")

In [ ]:
# ─────────────────────────────────────────────
# GRAPH — All Scores Comparison
# ─────────────────────────────────────────────
metrics_names = [
    'Accuracy', 'Macro\nPrecision', 'Macro\nRecall',
    'Macro F1', 'Weighted\nF1', 'Cohen\nKappa', 'Matthews\nMCC'
]

lda_vals = [
    metrics_lda['Accuracy'],    metrics_lda['Macro Precision'],
    metrics_lda['Macro Recall'], metrics_lda['Macro F1'],
    metrics_lda['Weighted F1'], metrics_lda['Cohen Kappa'],
    metrics_lda['Matthews MCC']
]
ab_vals = [
    metrics_ab['Accuracy'],    metrics_ab['Macro Precision'],
    metrics_ab['Macro Recall'], metrics_ab['Macro F1'],
    metrics_ab['Weighted F1'], metrics_ab['Cohen Kappa'],
    metrics_ab['Matthews MCC']
]
mb_vals = [
    metrics_mb['Accuracy'],    metrics_mb['Macro Precision'],
    metrics_mb['Macro Recall'], metrics_mb['Macro F1'],
    metrics_mb['Weighted F1'], metrics_mb['Cohen Kappa'],
    metrics_mb['Matthews MCC']
]

x     = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(18, 7))

bars1 = ax.bar(x - width,  lda_vals, width, label='LDA Baseline',
               color='#E67E22', alpha=0.9, edgecolor='white', linewidth=1.2)
bars2 = ax.bar(x,           ab_vals,  width, label='AraBERT',
               color='#3498DB', alpha=0.9, edgecolor='white', linewidth=1.2)
bars3 = ax.bar(x + width,   mb_vals,  width, label='MARBERT',
               color='#2ECC71', alpha=0.9, edgecolor='white', linewidth=1.2)

# Value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2,
                h + 0.008,
                f'{h:.3f}',
                ha='center', va='bottom',
                fontsize=8, fontweight='bold', rotation=90)



ax.set_ylabel('Score', fontsize=12)
ax.set_title('Full Metrics Comparison — LDA vs AraBERT vs MARBERT\n'
             '(Test Set, n=9,974)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=11, loc='lower right')

plt.tight_layout()
plt.savefig(save_path + 'graph_all_scores.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved graph_all_scores.png")

In [ ]:
# ─────────────────────────────────────────────
# GRAPH 1 — Overall Sentiment Distribution
# All 3 models side by side
# ─────────────────────────────────────────────
colors_sent = {'Negative': '#E74C3C', 'Neutral': '#95A5A6', 'Positive': '#2ECC71'}
classes     = ['Negative', 'Neutral', 'Positive']

# Get counts for all 3 models on test set
marbert_counts = pd.Series(pred_labels_mb).value_counts().reindex(classes, fill_value=0)
arabert_counts = pd.Series(pred_labels_ab).value_counts().reindex(classes, fill_value=0)
lda_counts     = pd.Series(lda_pred).value_counts().reindex(classes, fill_value=0)

x     = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 7))

bars1 = ax.bar(x - width, lda_counts,     width, label='LDA Baseline',
               color='#E67E22', alpha=0.9, edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x,          arabert_counts, width, label='AraBERT',
               color='#3498DB', alpha=0.9, edgecolor='white', linewidth=1.5)
bars3 = ax.bar(x + width,  marbert_counts, width, label='MARBERT',
               color='#2ECC71', alpha=0.9, edgecolor='white', linewidth=1.5)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2,
                h + 30,
                f'{h:,}\n({h/len(pred_labels_mb)*100:.1f}%)',
                ha='center', va='bottom',
                fontsize=9, fontweight='bold')

ax.set_ylabel('Number of Tweets', fontsize=12)
ax.set_title('Predicted Sentiment Distribution — LDA vs AraBERT vs MARBERT\n'
             '(Test Set, n=9,974)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=12)
ax.set_ylim(0, max(marbert_counts.max(),
                   arabert_counts.max(),
                   lda_counts.max()) * 1.25)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(save_path + 'graph1_sentiment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved graph1_sentiment_distribution.png")

In [ ]:
# ─────────────────────────────────────────────
# CHECK — All saved graphs and tables
# ─────────────────────────────────────────────
saved = os.listdir(save_path)

graphs = sorted([f for f in saved if f.startswith('graph')])
tables = sorted([f for f in saved if f.startswith('table')])

print(f"Graphs saved ({len(graphs)}):")
for g in graphs:
    size = os.path.getsize(os.path.join(save_path, g)) / 1e6
    print(f"  {g:<55} {size:.1f} MB")

print(f"\nTables saved ({len(tables)}):")
for t in tables:
    size = os.path.getsize(os.path.join(save_path, t)) / 1e6
    print(f"  {t:<55} {size:.1f} MB")